# Clase 11 — Evaluación de modelos
**Taller de Ciencia de Datos (EIN146A25) · 2026-2 · Jueves 22 de octubre**

Cuatro ejercicios sobre un mini-caso de **pedidos de un servicio de reparto a domicilio**: la pregunta es **"¿este pedido llegará con retraso?"**. Entrenar un modelo y partir los datos ya lo vieron en Machine Learning; hoy se aprende a **evaluarlo sin engañarse**: ¿con qué mido?, ¿cómo comparo?, ¿cómo ajusto? Al final, la plantilla de la sección 6 del informe (`notebooks/04-modelado.ipynb`) y la tabla de auditoría.

**Datos sintéticos generados para el lab: ninguna cifra describe a una empresa, un comercio o un reparto real.**

| # | Ejercicio | Qué se practica | Min |
|---|---|---|---|
| 1 | Métrica y baseline | costo del error, `DummyClassifier`, exactitud vs. sensibilidad vs. costo | 20 |
| 2 | Tres modelos, mismas particiones | `cross_validate` con los mismos folds, media ± desviación, empate práctico | 20 |
| 3 | Hiperparámetros: curva y búsqueda | `validation_curve`, `RandomizedSearchCV`, bordes del espacio; (extra) umbral por costo y test | 20 |
| 4 | Caza de la filtración | un resultado demasiado bueno: encontrar el paso que filtra y corregirlo | 30 |

Reglas del notebook:
- El archivo de práctica **se genera solo** en `data_practica/`: no hay que descargar nada y el lab no usa internet.
- Las celdas marcadas **✏️ TU TURNO** son las que deben completar.
- Cada ejercicio trae su **verificación automática** (debe imprimir todos los ✓) y una **solución propuesta** plegada: intenten primero sin mirarla.
- Todos los ejercicios parten de `X_tr`, `y_tr` y `CV` del Setup. Si uno no les sale, el siguiente puede usar las variables de referencia `_xxx` que deja la celda del profesor.
- **El test (`X_te`, `y_te`) no se toca** hasta el extra 3.3, y ahí una sola vez.
- `scikit-learn` es **obligatorio** en este lab. Ver `lab11-guia-evaluacion.md` para la referencia técnica de cada sección (§N).

In [ ]:
# Setup — ejecutar primero
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

try:
    import sklearn
except ImportError:
    print("scikit-learn no está instalado: ejecuten  pip install scikit-learn  y reinicien el kernel")
    raise ImportError("Este lab necesita scikit-learn: pip install scikit-learn") from None

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import (KFold, RandomizedSearchCV, StratifiedKFold, cross_val_predict,
                                     cross_validate, train_test_split, validation_curve)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 2026
CARPETA = Path("data_practica")
CARPETA.mkdir(exist_ok=True)
FIGS = Path("figs")


def guardar_figura(fig, nombre):
    """Guarda una figura en figs/ (la crea si no existe) con la resolución del informe."""
    FIGS.mkdir(exist_ok=True)
    ruta = FIGS / nombre
    fig.savefig(ruta, dpi=150, bbox_inches="tight")
    print("Guardado:", ruta)


def ejecutar_checks(checks):
    """checks: lista de (nombre, funcion_booleana_sin_argumentos).
    Atrapa NameError para que "Restart & Run All" con TU TURNO vacíos no se caiga."""
    faltan = 0
    for nombre, chk in checks:
        try:
            ok = bool(chk())
        except NameError as e:
            var = str(e).split("'")[1]
            print(f"✗ {nombre} (todavía no definen `{var}`)")
            faltan += 1
            continue
        except Exception as e:
            print(f"✗ {nombre} (error al evaluar: {e})")
            faltan += 1
            continue
        print(("✓" if ok else "✗"), nombre)
        faltan += not ok
    print("Todo OK" if faltan == 0 else f"Faltan {faltan}")
    return faltan == 0


print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· scikit-learn:", sklearn.__version__)
print("Carpeta de trabajo:", CARPETA.resolve())

In [ ]:
# Genera el archivo de práctica (no modificar esta celda)
rng = np.random.default_rng(2026)
N = 6000
N_COMERCIOS = 500

ZONAS = ["Centro", "Norte", "Sur", "Costa", "Cerro", "Periferia"]
EFECTO_ZONA = {"Centro": 0.0, "Norte": 0.1, "Sur": 0.15, "Costa": 0.35, "Cerro": 0.6, "Periferia": 0.5}
VEHICULOS = ["moto", "bici", "auto"]
CANALES = ["app", "web", "telefono"]

id_comercio = rng.integers(0, N_COMERCIOS, N)
efecto_comercio = rng.normal(0, 0.45, N_COMERCIOS)

zona = rng.choice(ZONAS, size=N, p=[0.28, 0.17, 0.17, 0.10, 0.10, 0.18])
distancia = np.clip(rng.gamma(2.2, 1.8, N) + 0.5, 0.5, 18).round(1)
n_items = np.clip(1 + rng.poisson(2.5, N), 1, 15)
_pesos_hora = np.array([3, 5, 9, 10, 5, 3, 3, 4, 7, 10, 10, 6, 3, 2])
hora = rng.choice(np.arange(10, 24), size=N, p=_pesos_hora / _pesos_hora.sum())
dia_semana = rng.integers(0, 7, N)
lluvia = (rng.random(N) < 0.2).astype(int)
punta = np.isin(hora, [13, 14, 20, 21]).astype(int)
trafico = np.clip(rng.normal(4.5 + 1.6 * punta + 0.9 * lluvia, 1.8, N), 0, 10).round(1)
prep = np.clip(rng.normal(12 + 1.3 * n_items, 4, N), 4, 60).round(0).astype(int)
antig = np.clip(rng.exponential(14, N), 0, 60).round(0).astype(int)
vehiculo = rng.choice(VEHICULOS, size=N, p=[0.60, 0.25, 0.15])
canal = rng.choice(CANALES, size=N, p=[0.70, 0.22, 0.08])

# riesgo de retraso (logístico): distancia, lluvia, hora punta, tráfico, preparación, zona, fin de semana,
# una interacción lluvia × punta, y un efecto propio de cada comercio
z = (-4.55 + 0.24 * distancia + 0.40 * lluvia + 0.50 * punta + 0.20 * (trafico - 4.5)
     + 0.07 * (prep - 15) - 0.02 * antig
     + 1.5 * np.array([EFECTO_ZONA[k] for k in zona])
     + 0.10 * distancia * (vehiculo == "bici") + 0.70 * lluvia * punta
     + 0.50 * (dia_semana >= 5) + efecto_comercio[id_comercio])
p = 1 / (1 + np.exp(-z))
retraso = (rng.random(N) < p).astype(int)

pedidos = pd.DataFrame({
    "id_pedido": [f"P{i:05d}" for i in range(1, N + 1)],
    "id_comercio": [f"C{c:03d}" for c in id_comercio],
    "zona": zona, "canal": canal, "tipo_vehiculo": vehiculo,
    "distancia_km": distancia, "n_items": n_items, "hora_pedido": hora, "dia_semana": dia_semana,
    "lluvia": lluvia, "trafico_idx": trafico, "tiempo_prep_min": prep,
    "antiguedad_repartidor_meses": antig, "retraso": retraso,
})
# id_pedido e id_comercio son texto: al leer, usar dtype=str
pedidos.to_csv(CARPETA / "pedidos_entregas.csv", index=False, encoding="utf-8")
print("pedidos_entregas.csv:", pedidos.shape)
print(f"Escrito en {CARPETA.resolve()} · tasa de retraso: {pedidos.retraso.mean():.1%}")

### Setup del modelado: datos, partición, costo y test

Esta celda la usan todos los ejercicios. `X` se arma **antes** de dividir porque `get_dummies` no aprende nada del objetivo (solo convierte categorías en columnas 0/1). El **test** (20 %) se separa **al inicio** y no se toca hasta el final. `CV` son los 5 folds estratificados que usarán **todos** los modelos.

> *No avisarle al cliente de un retraso (falso negativo) cuesta 10 unidades (reclamo y compensación); avisarle de un retraso que no ocurre (falso positivo) cuesta 1 unidad (un mensaje de más).*

Ver `lab11-guia-evaluacion.md`, §1, §3 y §8.

In [ ]:
df = pd.read_csv(CARPETA / "pedidos_entregas.csv", dtype={"id_pedido": str, "id_comercio": str})

X = pd.get_dummies(df.drop(columns=["id_pedido", "id_comercio", "retraso"]),
                   columns=["zona", "canal", "tipo_vehiculo"], dtype=int)   # 20 columnas
y = df["retraso"]
X_tr, X_te, y_tr, y_te, i_tr, i_te = train_test_split(
    X, y, np.arange(len(df)), test_size=0.2, stratify=y, random_state=SEED)
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

C_FN, C_FP = 10, 1


def costo_total(y_real, y_pred, c_fn=C_FN, c_fp=C_FP):
    """Costo del error: c_fn por cada retraso no avisado + c_fp por cada aviso de más."""
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    return c_fn * fn + c_fp * fp


_USOS_TEST = {"n": 0}


def evaluar_test(modelo_ajustado, umbral=0.5):
    """Evalúa en el TEST. Solo debería llamarse UNA vez, con el modelo ya elegido."""
    _USOS_TEST["n"] += 1
    if _USOS_TEST["n"] > 1:
        print(f"⚠ Es la vez N° {_USOS_TEST['n']} que usan el test: un test usado varias veces deja de ser un test.")
    p = modelo_ajustado.predict_proba(X_te)[:, 1]
    pred = (p >= umbral).astype(int)
    return {"umbral": umbral,
            "sensibilidad": recall_score(y_te, pred),
            "precision": precision_score(y_te, pred, zero_division=0),
            "pr_auc": average_precision_score(y_te, p),
            "roc_auc": roc_auc_score(y_te, p),
            "costo_por_pedido": costo_total(y_te, pred) / len(y_te)}


print(f"X: {X.shape[0]} filas × {X.shape[1]} columnas")
print(f"Entrenamiento: {len(X_tr)} filas ({y_tr.sum()} retrasos, {y_tr.mean():.1%}) · "
      f"test: {len(X_te)} filas ({y_te.sum()} retrasos)")

Aquí el escalado ya va dentro del modelo con `make_pipeline`: así se ajusta solo con el entrenamiento de cada fold. La próxima clase aprendemos a escribir esto; hoy solo lo usamos.

`crear_modelos()` devuelve **modelos nuevos** (sin ajustar) cada vez que se llama: tres familias distintas (lineal, árbol, ensamble).

In [ ]:
def crear_modelos():
    return {
        "Regresión logística": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        "Árbol": DecisionTreeClassifier(random_state=SEED),
        "Bosque aleatorio": RandomForestClassifier(n_estimators=200, min_samples_leaf=10,
                                                   random_state=SEED, n_jobs=-1),
    }


crear_modelos()

---
## Ejercicio 1 — Métrica y baseline (≈20 min)

Antes de entrenar nada hay que decidir **con qué se mide**. La métrica no se elige por costumbre: sale de **qué error es más caro**. Y todo modelo se compara contra lo que se logra **sin modelo** (una política trivial): si no le gana, no sirve.

Ver `lab11-guia-evaluacion.md`, §1–2.

### 1.1 Primero, el error
Un "modelo" que predice **siempre 0** (nunca hay retraso), evaluado con la exactitud.

In [ ]:
pred_nunca = np.zeros(len(y_tr), dtype=int)
exactitud = accuracy_score(y_tr, pred_nunca)
print(f"¿Un {exactitud * 100:.1f} % de aciertos? Pregunta: ¿le serviría a la empresa?".replace(".", ","))
print(f"Sensibilidad (retrasos que detecta): {recall_score(y_tr, pred_nunca):.3f}")

*Acierta casi 9 de cada 10 pedidos y no detecta ni un solo retraso. ¿Cuánto le cuesta eso a la empresa?*

### 1.2 ✏️ TU TURNO
Sobre el **entrenamiento** (`X_tr`, `y_tr`):

1. `frase_costo` = una frase (texto) con **qué error es más caro y por qué**.
2. `baseline` = diccionario con `exactitud`, `sensibilidad` y `costo_por_pedido` de un `DummyClassifier(strategy="most_frequent")` ajustado sobre `X_tr, y_tr` (el costo con `costo_total`, dividido por el número de pedidos).
3. `costo_siempre` = costo por pedido de la política **"avisar siempre"** (predecir 1 en todos: no es un `Dummy`, se calcula a mano).
4. `pr_auc_baseline` = PR-AUC del baseline = **prevalencia** de retrasos en el entrenamiento.
5. `metrica_elegida` = una de `"exactitud"`, `"sensibilidad"`, `"f2"`, `"pr_auc"`, `"costo"`, y `justificacion` = una línea que la justifique según `frase_costo`.
6. En la celda de texto de abajo: ¿el baseline "nunca avisar" es el piso correcto?
7. `conclusion_ej1` = `"a"`, `"b"` o `"c"`:
   - a) El baseline acierta el 88,6 %: un modelo que no supere ese 88,6 % de exactitud no sirve.
   - b) La exactitud engaña con clase desbalanceada: hay que elegir la métrica según el costo del error, y el baseline más exigente es "avisar siempre".
   - c) Como los retrasos son solo el 11,4 %, conviene maximizar la sensibilidad a cualquier costo: "avisar siempre" es la mejor política posible.

Pistas: `np.ones(len(y_tr), dtype=int)` · `y_tr.mean()` · guía §1 (tabla de métricas) y §2 (baselines).

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# frase_costo = "..."
#
# dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
# pred = dummy.predict(X_tr)
# baseline = {"exactitud": ...,
#             "sensibilidad": ...,
#             "costo_por_pedido": costo_total(y_tr, pred) / len(y_tr)}
# costo_siempre = ...
# pr_auc_baseline = ...
#
# metrica_elegida = "..."      # "exactitud", "sensibilidad", "f2", "pr_auc" o "costo"
# justificacion = "..."
# conclusion_ej1 = ...

✏️ **Respuesta (6):** ¿el baseline "nunca avisar" es el piso correcto? *(escriban aquí, con números)*

### ✅ Verificación automática
Ejecuten esta celda cuando tengan todo lo de arriba. Debe imprimir 8 ✓.

In [ ]:
def verificar_ej1():
    checks = [
        ("frase_costo tiene al menos 20 caracteres",
         lambda: isinstance(frase_costo, str) and len(frase_costo.strip()) >= 20),
        ('baseline["exactitud"] ≈ 0,886 (±0,01)',
         lambda: abs(baseline["exactitud"] - 0.886) <= 0.01),
        ('baseline["sensibilidad"] == 0',
         lambda: baseline["sensibilidad"] == 0),
        ('baseline["costo_por_pedido"] ≈ 1,144 (±0,03)',
         lambda: abs(baseline["costo_por_pedido"] - 1.144) <= 0.03),
        ("costo_siempre ≈ 0,886 (±0,03)",
         lambda: abs(costo_siempre - 0.886) <= 0.03),
        ("pr_auc_baseline ≈ 0,114 (±0,01)",
         lambda: abs(pr_auc_baseline - 0.114) <= 0.01),
        ('metrica_elegida es una opción válida y no es "exactitud"',
         lambda: metrica_elegida in {"sensibilidad", "f2", "pr_auc", "costo"}),
        ("conclusion_ej1 es la correcta",
         lambda: conclusion_ej1 == "b"),
    ]
    return ejecutar_checks(checks)

verificar_ej1()

<details>
<summary><b>Solución propuesta (Ejercicio 1)</b> — clic para desplegar</summary>

```python
frase_costo = ("No avisar un retraso que sí ocurre (falso negativo) cuesta 10 veces más que avisar uno que no "
               "ocurre (falso positivo): un reclamo con compensación contra un mensaje de más.")

dummy = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr)
pred = dummy.predict(X_tr)
baseline = {"exactitud": accuracy_score(y_tr, pred),
            "sensibilidad": recall_score(y_tr, pred),
            "costo_por_pedido": costo_total(y_tr, pred) / len(y_tr)}
costo_siempre = costo_total(y_tr, np.ones(len(y_tr), dtype=int)) / len(y_tr)
pr_auc_baseline = y_tr.mean()

metrica_elegida = "costo"
justificacion = ("el falso negativo cuesta 10 veces el falso positivo: se mide el costo por pedido; "
                 "PR-AUC como secundaria para comparar modelos sin fijar umbral")
conclusion_ej1 = "b"

print({k: round(float(v), 3) for k, v in baseline.items()})
print(f"avisar siempre: {costo_siempre:.3f} por pedido · PR-AUC del baseline: {pr_auc_baseline:.3f}")
verificar_ej1()
```

"Nunca avisar" acierta el **88,6 %** de los pedidos, pero su sensibilidad es **0**: deja pasar los 549 retrasos del entrenamiento y cuesta **1,144** unidades por pedido (5.490 en total). "Avisar siempre" acierta solo el 11,4 % y, sin embargo, cuesta **0,886** por pedido (4.251 mensajes de más): **es mejor que "nunca avisar"**. Con un costo 10 a 1, la exactitud premia justo la política más cara.

Por eso el piso real no es 1,144 sino **0,886**: un modelo que no baje de 0,886 por pedido no le sirve a la empresa, aunque tenga 90 % de exactitud. La métrica principal es el **costo por pedido** (necesita un umbral; ver el extra 3.3) y, para comparar modelos sin fijar umbral, la **PR-AUC**, cuyo valor sin modelo es la prevalencia (**0,114**). F2 (sensibilidad pesa más que precisión) también es defendible; la exactitud no.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_pred0 = DummyClassifier(strategy="most_frequent").fit(X_tr, y_tr).predict(X_tr)
_baseline = {"exactitud": accuracy_score(y_tr, _pred0),
             "sensibilidad": recall_score(y_tr, _pred0),
             "costo_por_pedido": costo_total(y_tr, _pred0) / len(y_tr)}
_costo_siempre = costo_total(y_tr, np.ones(len(y_tr), dtype=int)) / len(y_tr)
_pr_auc_baseline = y_tr.mean()

assert (len(X_tr), int(y_tr.sum())) == (4800, 549)
assert _baseline["sensibilidad"] == 0 and _costo_siempre < _baseline["costo_por_pedido"], \
    "La solución de referencia del Ej. 1 no pasa la verificación"
print({k: round(float(v), 3) for k, v in _baseline.items()})
print(f"avisar siempre: {_costo_siempre:.3f} por pedido · PR-AUC del baseline: {_pr_auc_baseline:.3f}")

---
## Ejercicio 2 — Tres modelos, mismas particiones (≈20 min)

Tres modelos de familias distintas (`crear_modelos()`): ¿cuál es mejor? Para que la comparación sea justa, los tres se evalúan **en los mismos 5 folds** (`CV`), y la diferencia entre ellos se compara con **cuánto varía la métrica de un fold a otro**.

Ver `lab11-guia-evaluacion.md`, §3–4.

### 2.1 Primero, el error
Tres corridas. En cada una, **cada modelo se evalúa con su propia partición** (un `KFold` con otra semilla: 1, 2 o 3) y en **un solo fold** de validación, con PR-AUC.

In [ ]:
def puntaje_un_fold(modelo, semilla):
    """MAL a propósito: cada modelo con SU partición, y un solo fold de validación."""
    a, b = next(KFold(n_splits=5, shuffle=True, random_state=semilla).split(X_tr))
    m = clone(modelo).fit(X_tr.iloc[a], y_tr.iloc[a])
    return average_precision_score(y_tr.iloc[b], m.predict_proba(X_tr.iloc[b])[:, 1])


for corrida in range(3):
    res = {nombre: puntaje_un_fold(m, semilla=(j + corrida) % 3 + 1)
           for j, (nombre, m) in enumerate(crear_modelos().items())}
    ranking = "  >  ".join(f"{k} ({v:.3f})" for k, v in sorted(res.items(), key=lambda t: -t[1]))
    print(f"Corrida {corrida + 1}: {ranking}")

*¿Cuál es el mejor? ¿Qué cambió entre corridas: los modelos o las particiones?*

### 2.2 ✏️ TU TURNO
Con `CV` y `crear_modelos()`:

1. `scores_ap` = diccionario modelo → arreglo con las **5 PR-AUC** (una por fold). Usen `cross_validate` con `scoring={"ap": "average_precision", "roc": "roc_auc"}`, `cv=CV` y `n_jobs=-1`.
2. `tabla_cv` = DataFrame con índice `modelo` y columnas `ap_media`, `ap_std`, `roc_media`, `roc_std`, más una fila `Baseline` con `ap_media = _pr_auc_baseline` (y `roc_media = 0,5`).
3. Gráfico de **cajas** de `scores_ap` por modelo con una línea horizontal en el baseline; guárdenlo con `guardar_figura(fig2, "lab11_cv_cajas.png")`.
4. `veredicto` (`"ganador"` o `"empate"`) y `elegido` (nombre del modelo), con la regla de la S8 de la cátedra: **si la diferencia entre el primero y el segundo es menor que la desviación entre folds, es empate práctico y gana el más simple**.
5. En la celda de texto de abajo: ¿la diferencia entre el primero y el segundo es mayor que la desviación entre folds?
6. `conclusion_ej2` = `"a"`, `"b"` o `"c"`:
   - a) La regresión logística gana: tiene la PR-AUC media más alta, así que es mejor que el bosque.
   - b) El bosque aleatorio es el mejor: es el modelo más flexible y la diferencia con la logística es pequeña.
   - c) Logística y bosque empatan: la diferencia es menor que la desviación entre folds, así que se elige la más simple; el árbol sin límite es claramente peor.

Pistas: `r = cross_validate(...)` devuelve `r["test_ap"]` y `r["test_roc"]` · `pd.DataFrame(filas).set_index("modelo")` · `ax.boxplot(list(scores_ap.values()))` y `ax.set_xticks(range(1, 4), list(scores_ap))`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# scores_ap, filas = {}, []
# for nombre, modelo in crear_modelos().items():
#     r = cross_validate(modelo, X_tr, y_tr, cv=CV, n_jobs=-1,
#                        scoring={"ap": "average_precision", "roc": "roc_auc"})
#     scores_ap[nombre] = ...
#     filas.append({"modelo": nombre, "ap_media": ..., "ap_std": ..., "roc_media": ..., "roc_std": ...})
# filas.append({"modelo": "Baseline", "ap_media": _pr_auc_baseline, "ap_std": 0.0, "roc_media": 0.5, "roc_std": 0.0})
# tabla_cv = pd.DataFrame(filas).set_index("modelo")
#
# fig2, ax2 = plt.subplots(figsize=(7, 4))
# ax2.boxplot(...)
# ax2.set_xticks(range(1, 4), list(scores_ap))
# ax2.axhline(...)
# ax2.set_ylabel(...); ax2.set_title(...)
# guardar_figura(fig2, "lab11_cv_cajas.png")
#
# veredicto = ...
# elegido = ...
# conclusion_ej2 = ...

✏️ **Respuesta (5):** ¿la diferencia entre el primero y el segundo es mayor que la desviación entre folds? *(escriban aquí, con los dos números)*

### ✅ Verificación automática
Debe imprimir 9 ✓.

In [ ]:
_MODELOS = ["Regresión logística", "Árbol", "Bosque aleatorio"]


def _tabla(t):
    return t.set_index("modelo") if "modelo" in t.columns else t


def verificar_ej2():
    checks = [
        ("tabla_cv tiene los tres modelos y la fila Baseline",
         lambda: set(_MODELOS + ["Baseline"]) <= set(_tabla(tabla_cv).index)),
        ("ap_media ≈ 0,352 (logística), 0,131 (árbol), 0,337 (bosque) (±0,02)",
         lambda: all(abs(_tabla(tabla_cv).loc[m, "ap_media"] - v) <= 0.02
                     for m, v in zip(_MODELOS, [0.352, 0.131, 0.337]))),
        ("ap_std de los tres modelos entre 0,005 y 0,05",
         lambda: all(0.005 <= _tabla(tabla_cv).loc[m, "ap_std"] <= 0.05 for m in _MODELOS)),
        ("el árbol queda al menos 0,01 sobre el baseline y al menos 0,15 bajo la logística",
         lambda: (_tabla(tabla_cv).loc["Árbol", "ap_media"] - _tabla(tabla_cv).loc["Baseline", "ap_media"] >= 0.01
                  and _tabla(tabla_cv).loc["Regresión logística", "ap_media"]
                  - _tabla(tabla_cv).loc["Árbol", "ap_media"] >= 0.15)),
        ("scores_ap tiene 5 valores por modelo",
         lambda: set(_MODELOS) <= set(scores_ap) and all(len(scores_ap[m]) == 5 for m in _MODELOS)),
        ('veredicto == "empate"',
         lambda: veredicto == "empate"),
        ('elegido == "Regresión logística"',
         lambda: elegido == "Regresión logística"),
        ("existe figs/lab11_cv_cajas.png",
         lambda: (FIGS / "lab11_cv_cajas.png").exists()),
        ("conclusion_ej2 es la correcta",
         lambda: conclusion_ej2 == "c"),
    ]
    return ejecutar_checks(checks)

verificar_ej2()

<details>
<summary><b>Solución propuesta (Ejercicio 2)</b> — clic para desplegar</summary>

```python
scores_ap, filas = {}, []
for nombre, modelo in crear_modelos().items():
    r = cross_validate(modelo, X_tr, y_tr, cv=CV, n_jobs=-1,
                       scoring={"ap": "average_precision", "roc": "roc_auc"})
    scores_ap[nombre] = r["test_ap"]
    filas.append({"modelo": nombre, "ap_media": r["test_ap"].mean(), "ap_std": r["test_ap"].std(),
                  "roc_media": r["test_roc"].mean(), "roc_std": r["test_roc"].std()})
filas.append({"modelo": "Baseline", "ap_media": _pr_auc_baseline, "ap_std": 0.0, "roc_media": 0.5, "roc_std": 0.0})
tabla_cv = pd.DataFrame(filas).set_index("modelo")
display(tabla_cv.round(3))

fig2, ax2 = plt.subplots(figsize=(7, 4))
ax2.boxplot(list(scores_ap.values()))
ax2.set_xticks(range(1, len(scores_ap) + 1), list(scores_ap))
ax2.axhline(_pr_auc_baseline, color="gray", linestyle="--", label=f"baseline ({_pr_auc_baseline:.3f})")
ax2.set_ylabel("PR-AUC por fold (5 folds)")
ax2.set_title("Logística y bosque se solapan; el árbol sin límite apenas supera al baseline")
ax2.legend()
guardar_figura(fig2, "lab11_cv_cajas.png")

orden = tabla_cv.drop("Baseline").sort_values("ap_media", ascending=False)
diferencia = orden.ap_media.iloc[0] - orden.ap_media.iloc[1]
desviacion = orden.ap_std.iloc[:2].max()
veredicto = "empate" if diferencia < desviacion else "ganador"
elegido = "Regresión logística"      # empate con el bosque: gana el más simple
print(f"{orden.index[0]} − {orden.index[1]} = {diferencia:.3f}  vs.  desviación entre folds ≈ {desviacion:.3f}"
      f"  →  {veredicto}; elegido: {elegido}")
conclusion_ej2 = "c"
verificar_ej2()
```

Con los **mismos 5 folds** para los tres: regresión logística **0,352 ± 0,027**, bosque **0,337 ± 0,025**, árbol sin límite **0,131 ± 0,009** (baseline 0,114). La diferencia entre los dos primeros es **0,015**, menor que la desviación entre folds (≈ 0,027): las cajas se solapan y es un **empate práctico**. A igualdad, gana el más simple: la **regresión logística** (lineal, rápida, sus coeficientes se pueden explicar). El árbol sin límite memoriza el entrenamiento y apenas supera al baseline (ROC-AUC ≈ 0,55).

En la 2.1 cada modelo tuvo su propia partición y un solo fold: la logística "valía" 0,355, 0,384 o 0,348 según la partición que le tocó, y en la tercera corrida el bosque "ganaba". Eso no era una diferencia entre modelos sino **ruido de la partición**. Con los mismos folds, el ruido afecta a todos por igual y la media ± desviación dice cuánto creerle a la diferencia.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
_scores_ap, _filas = {}, []
for _nombre, _m in crear_modelos().items():
    _r = cross_validate(_m, X_tr, y_tr, cv=CV, n_jobs=-1, scoring={"ap": "average_precision", "roc": "roc_auc"})
    _scores_ap[_nombre] = _r["test_ap"]
    _filas.append({"modelo": _nombre, "ap_media": _r["test_ap"].mean(), "ap_std": _r["test_ap"].std(),
                   "roc_media": _r["test_roc"].mean(), "roc_std": _r["test_roc"].std()})
_filas.append({"modelo": "Baseline", "ap_media": _pr_auc_baseline, "ap_std": 0.0, "roc_media": 0.5, "roc_std": 0.0})
_tabla_cv = pd.DataFrame(_filas).set_index("modelo")
_elegido = "Regresión logística"

_dif = _tabla_cv.loc["Regresión logística", "ap_media"] - _tabla_cv.loc["Bosque aleatorio", "ap_media"]
assert 0 < _dif < _tabla_cv.loc[["Regresión logística", "Bosque aleatorio"], "ap_std"].max(), \
    "La solución de referencia del Ej. 2 no pasa la verificación"
print(_tabla_cv.round(3).to_string())

---
## Ejercicio 3 — Hiperparámetros: curva y búsqueda (≈20 min)

Un hiperparámetro de **complejidad** (la profundidad de un árbol, el tamaño mínimo de las hojas) decide cuánto puede memorizar el modelo. La **curva de validación** muestra dónde se pasa de complejidad; la **búsqueda** prueba combinaciones, pero **más búsqueda no garantiza un mejor modelo**.

Ver `lab11-guia-evaluacion.md`, §5–6.

### 3.1 Primero, el error
Un árbol de profundidad 20 evaluado (i) **en el mismo entrenamiento** con que se ajustó y (ii) con validación cruzada.

In [ ]:
arbol20 = DecisionTreeClassifier(max_depth=20, random_state=SEED).fit(X_tr, y_tr)
auc_mismo = roc_auc_score(y_tr, arbol20.predict_proba(X_tr)[:, 1])
auc_cv = cross_validate(DecisionTreeClassifier(max_depth=20, random_state=SEED), X_tr, y_tr,
                        cv=CV, scoring="roc_auc", n_jobs=-1)["test_score"].mean()
print(f"(i)  ROC-AUC en el mismo entrenamiento: {auc_mismo:.3f}   ¡perfecto!")
print(f"(ii) ROC-AUC del mismo árbol con CV:    {auc_cv:.3f}")

*¿Cuál de los dos números le cuenta a la empresa algo cierto?*

### 3.2 ✏️ TU TURNO

1. `curva` = DataFrame con columnas `profundidad` (1 a 20), `entrenamiento` y `validacion`: las **medias** de `validation_curve` de un `DecisionTreeClassifier(random_state=SEED)` en `max_depth`, con ROC-AUC y `cv=CV`. Gráfico con las dos curvas y las **tres zonas rotuladas** (subajuste / buen punto / sobreajuste); guárdenlo como `lab11_curva_validacion.png`.
2. `prof_optima` = profundidad con la mayor `validacion`, y `sobreajuste_desde` = la menor profundidad **mayor que `prof_optima`** en que `validacion` cae **más de 0,03** bajo su máximo.
3. Con el `espacio` ya dado, `busqueda = RandomizedSearchCV(RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1), espacio, n_iter=15, scoring="average_precision", cv=CV, random_state=SEED)`, ajustada sobre `X_tr, y_tr` (≈ 10–30 s).
4. `mejor_params`, `mejor_ap` y `mejor_std` (= `busqueda.cv_results_["std_test_score"][busqueda.best_index_]`).
5. `en_borde` = diccionario `parámetro → True/False`: ¿el mejor valor es el **mínimo o el máximo** de su lista en `espacio`?
6. `diferencia_vs_defecto` = `mejor_ap` − `ap_media` del bosque por defecto del Ejercicio 2 (`_tabla_cv.loc["Bosque aleatorio", "ap_media"]`).
7. En la celda de texto de abajo: ¿mejoró algo la búsqueda? ¿Hay que ampliar el rango de algún hiperparámetro?
8. `conclusion_ej3` = `"a"`, `"b"` o `"c"`:
   - a) La curva muestra dónde se pasa de complejidad; la búsqueda no mejoró más que la desviación entre folds, así que no hay evidencia de que valga la pena el modelo más complejo.
   - b) La búsqueda encontró la mejor combinación: hay que reemplazar el bosque por defecto por el bosque afinado y elegirlo como modelo final.
   - c) El árbol de profundidad 20 es el mejor de la curva, porque su ROC-AUC de entrenamiento es 1,000.

Pistas: `tr, va = validation_curve(..., param_name="max_depth", param_range=profundidades, ...)` y `tr.mean(axis=1)` · `ax.axvspan(x0, x1, alpha=0.1)` pinta una zona · `curva.loc[curva.validacion.idxmax(), "profundidad"]`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# profundidades = list(range(1, 21))
# tr, va = validation_curve(DecisionTreeClassifier(random_state=SEED), X_tr, y_tr, param_name="max_depth",
#                           param_range=profundidades, cv=CV, scoring="roc_auc", n_jobs=-1)
# curva = pd.DataFrame({"profundidad": profundidades, "entrenamiento": ..., "validacion": ...})
# prof_optima = ...
# sobreajuste_desde = ...
#
# fig3, ax3 = plt.subplots(figsize=(8, 4.5))
# ax3.plot(...); ax3.plot(...)
# ax3.axvspan(...)            # tres zonas, cada una con su texto
# ax3.set_xlabel(...); ax3.set_ylabel(...); ax3.set_title(...); ax3.legend()
# guardar_figura(fig3, "lab11_curva_validacion.png")
#
espacio = {"max_depth": list(range(3, 13)),                  # 10 valores
           "min_samples_leaf": [1, 3, 5, 10, 20, 30, 50],    # 7 valores
           "max_features": [0.2, 0.3, 0.5, 0.7, 1.0]}        # 5 valores → 350 combinaciones posibles
# busqueda = RandomizedSearchCV(RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1),
#                               espacio, n_iter=15, scoring="average_precision", cv=CV, random_state=SEED)
# busqueda.fit(X_tr, y_tr)
# mejor_params = ...
# mejor_ap = ...
# mejor_std = ...
# en_borde = {k: ... for k, v in mejor_params.items()}
# diferencia_vs_defecto = mejor_ap - _tabla_cv.loc["Bosque aleatorio", "ap_media"]
# conclusion_ej3 = ...

✏️ **Respuesta (7):** ¿mejoró algo la búsqueda? ¿Hay que ampliar el rango de algún hiperparámetro? *(escriban aquí)*

### ✅ Verificación automática
Debe imprimir 10 ✓.

In [ ]:
def verificar_ej3():
    checks = [
        ("curva tiene 20 filas (profundidades 1 a 20)",
         lambda: len(curva) == 20 and list(curva.profundidad) == list(range(1, 21))),
        ("profundidad 20: entrenamiento ≥ 0,99 y validación ≤ 0,60",
         lambda: curva.set_index("profundidad").loc[20, "entrenamiento"] >= 0.99
                 and curva.set_index("profundidad").loc[20, "validacion"] <= 0.60),
        ("prof_optima entre 3 y 6",
         lambda: 3 <= prof_optima <= 6),
        ("sobreajuste_desde entre 6 y 9",
         lambda: 6 <= sobreajuste_desde <= 9),
        ("existe figs/lab11_curva_validacion.png",
         lambda: (FIGS / "lab11_curva_validacion.png").exists()),
        ('busqueda ajustada con 15 combinaciones y scoring="average_precision"',
         lambda: len(busqueda.cv_results_["params"]) == 15 and busqueda.scoring == "average_precision"),
        ("mejor_ap ≈ 0,334 (±0,02)",
         lambda: abs(mejor_ap - 0.334) <= 0.02),
        ("en_borde tiene las 3 claves y es consistente con mejor_params y espacio",
         lambda: set(en_borde) == set(espacio)
                 and all(en_borde[k] == (mejor_params[k] in (min(espacio[k]), max(espacio[k]))) for k in espacio)),
        ("|diferencia_vs_defecto| < mejor_std (la búsqueda no supera al bosque por defecto más allá del ruido)",
         lambda: abs(diferencia_vs_defecto) < mejor_std),
        ("conclusion_ej3 es la correcta",
         lambda: conclusion_ej3 == "a"),
    ]
    return ejecutar_checks(checks)

verificar_ej3()

<details>
<summary><b>Solución propuesta (Ejercicio 3)</b> — clic para desplegar</summary>

```python
profundidades = list(range(1, 21))
tr, va = validation_curve(DecisionTreeClassifier(random_state=SEED), X_tr, y_tr, param_name="max_depth",
                          param_range=profundidades, cv=CV, scoring="roc_auc", n_jobs=-1)
curva = pd.DataFrame({"profundidad": profundidades, "entrenamiento": tr.mean(axis=1), "validacion": va.mean(axis=1)})
prof_optima = int(curva.loc[curva.validacion.idxmax(), "profundidad"])
cae = (curva.profundidad > prof_optima) & (curva.validacion < curva.validacion.max() - 0.03)
sobreajuste_desde = int(curva.loc[cae, "profundidad"].min())

fig3, ax3 = plt.subplots(figsize=(8, 4.5))
ax3.plot(curva.profundidad, curva.entrenamiento, marker="o", label="entrenamiento")
ax3.plot(curva.profundidad, curva.validacion, marker="o", label="validación (CV)")
ax3.axvspan(0.5, 2.5, color="tab:blue", alpha=0.08)
ax3.axvspan(2.5, sobreajuste_desde - 0.5, color="tab:green", alpha=0.12)
ax3.axvspan(sobreajuste_desde - 0.5, 20.5, color="tab:red", alpha=0.08)
for x, txt in [(1.5, "subajuste"), ((2 + sobreajuste_desde) / 2, "buen punto"), (14, "sobreajuste")]:
    ax3.text(x, 0.515, txt, ha="center", fontsize=9)
ax3.set_ylim(0.5, 1.02)
ax3.set_xticks(profundidades)
ax3.set_xlabel("max_depth del árbol"); ax3.set_ylabel("ROC-AUC (media de 5 folds)")
ax3.set_title(f"El árbol mejora hasta profundidad {prof_optima} y sobreajusta desde la {sobreajuste_desde}")
ax3.legend(loc="center right")
guardar_figura(fig3, "lab11_curva_validacion.png")

espacio = {"max_depth": list(range(3, 13)),                  # 10 valores
           "min_samples_leaf": [1, 3, 5, 10, 20, 30, 50],    # 7 valores
           "max_features": [0.2, 0.3, 0.5, 0.7, 1.0]}        # 5 valores → 350 combinaciones posibles
busqueda = RandomizedSearchCV(RandomForestClassifier(n_estimators=150, random_state=SEED, n_jobs=-1),
                              espacio, n_iter=15, scoring="average_precision", cv=CV, random_state=SEED)
busqueda.fit(X_tr, y_tr)
mejor_params = busqueda.best_params_
mejor_ap = busqueda.best_score_
mejor_std = busqueda.cv_results_["std_test_score"][busqueda.best_index_]
en_borde = {k: v in (min(espacio[k]), max(espacio[k])) for k, v in mejor_params.items()}
diferencia_vs_defecto = mejor_ap - _tabla_cv.loc["Bosque aleatorio", "ap_media"]

print("Mejor combinación:", mejor_params, f"→ PR-AUC {mejor_ap:.3f} ± {mejor_std:.3f}")
print("En el borde del espacio:", en_borde)
print(f"Diferencia con el bosque por defecto: {diferencia_vs_defecto:+.3f} (desviación entre folds {mejor_std:.3f})")
conclusion_ej3 = "a"
verificar_ej3()
```

**Curva.** Con profundidad 1–2 el árbol es demasiado simple (**subajuste**: entrenamiento y validación bajos, 0,61 y 0,67). La validación llega a su máximo en profundidad **4 (0,688)**; 3 a 5 son prácticamente iguales. Desde la **7** la validación cae más de 0,03 bajo su máximo mientras el entrenamiento sigue subiendo: **sobreajuste**. En la 20 el entrenamiento es **1,000** y la validación **0,547**: el árbol memoriza los 4.800 pedidos y casi no generaliza. La brecha entre las dos curvas es la medida del sobreajuste. Por eso el 1,000 de la 3.1 no le dice nada cierto a la empresa: es un examen con las respuestas a la vista.

**Búsqueda.** 15 combinaciones de 350 posibles (≈ 4 %), con PR-AUC y la misma `CV`. La mejor da **0,334 ± 0,027**, prácticamente lo mismo que el bosque por defecto del Ej. 2 (0,337): **no mejoró nada más allá del ruido**. Además, `max_features` quedó en su borde (0,2 es el mínimo de la lista): si se quisiera seguir buscando, habría que **ampliar el rango hacia abajo** (0,05, 0,1), porque el óptimo podría estar fuera del espacio probado. (Los parámetros exactos pueden variar con la versión de scikit-learn; lo que no cambia es la lección.) La regresión logística sigue siendo el modelo elegido.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
# La búsqueda no se repite aquí (tarda 10–30 s y ningún ejercicio posterior la usa).
_tr, _va = validation_curve(DecisionTreeClassifier(random_state=SEED), X_tr, y_tr, param_name="max_depth",
                            param_range=list(range(1, 21)), cv=CV, scoring="roc_auc", n_jobs=-1)
_curva = pd.DataFrame({"profundidad": range(1, 21), "entrenamiento": _tr.mean(axis=1), "validacion": _va.mean(axis=1)})
_prof_optima = int(_curva.loc[_curva.validacion.idxmax(), "profundidad"])

assert 3 <= _prof_optima <= 6 and _curva.entrenamiento.iloc[-1] >= 0.99, \
    "La solución de referencia del Ej. 3 no pasa la verificación"
print(_curva.set_index("profundidad").loc[[1, 2, 4, 7, 10, 20]].round(3).T.to_string())

### 3.3 Extra (opcional, para la casa o si sobra tiempo) · ✏️ Umbral por costo y reporte en el test

Un clasificador entrega **probabilidades**; el umbral 0,5 es solo un valor por defecto. Con un falso negativo 10 veces más caro, conviene avisar con probabilidades mucho menores. En teoría, con probabilidades bien calibradas, el umbral que minimiza el costo es `C_FP / (C_FP + C_FN)` = **0,09** (guía §8).

1. Con el modelo elegido en el Ejercicio 2 (`elegido`, o `_elegido`): `p_oof = cross_val_predict(modelo, X_tr, y_tr, cv=CV, method="predict_proba")[:, 1]` (probabilidades **fuera de fold**: cada pedido predicho por un modelo que no lo vio).
2. Barrido de umbrales de 0,02 a 0,69 (pasos de 0,01) y `umbral_opt` = el que minimiza `costo_total` sobre el entrenamiento.
3. Ajustar el modelo con **todo** el entrenamiento y llamar **una sola vez** a `evaluar_test(modelo, umbral_opt)`; guardar en `resultado_test`.
4. Comparar, en el test, con el umbral 0,5 y con las dos políticas triviales.

In [ ]:
# ✏️ TU TURNO (opcional) — completen (descomenten y reemplacen los ...)
# modelo_final = crear_modelos()[elegido]          # o _elegido
# p_oof = cross_val_predict(modelo_final, X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
# umbrales = np.round(np.arange(0.02, 0.70, 0.01), 2)
# costos = np.array([... for u in umbrales])
# umbral_opt = ...
#
# modelo_final.fit(X_tr, y_tr)
# resultado_test = evaluar_test(modelo_final, umbral_opt)      # UNA sola vez
#
# p_te = modelo_final.predict_proba(X_te)[:, 1]                # para comparar con 0,5 y las triviales
# ...

### ✅ Verificación automática (extra)
Debe imprimir 4 ✓, o `⊘` si no hicieron el extra (no cuenta como falla).

In [ ]:
def verificar_ej3_extra():
    if "umbral_opt" not in globals():
        print("⊘ extra no realizado (no cuenta como falla)")
        return True
    checks = [
        ("umbral_opt entre 0,05 y 0,20",
         lambda: 0.05 <= umbral_opt <= 0.20),
        ("el test se usó una sola vez (_USOS_TEST == 1)",
         lambda: _USOS_TEST["n"] == 1),
        ("costo por pedido en el test < 0,886 (mejor que avisar siempre) y ≤ 0,70",
         lambda: resultado_test["costo_por_pedido"] < 0.886 and resultado_test["costo_por_pedido"] <= 0.70),
        ("sensibilidad en el test ≥ 0,60",
         lambda: resultado_test["sensibilidad"] >= 0.60),
    ]
    return ejecutar_checks(checks)

verificar_ej3_extra()

<details>
<summary><b>Solución propuesta (Ejercicio 3, extra 3.3)</b> — clic para desplegar</summary>

```python
modelo_final = crear_modelos()[_elegido]
p_oof = cross_val_predict(modelo_final, X_tr, y_tr, cv=CV, method="predict_proba", n_jobs=-1)[:, 1]
umbrales = np.round(np.arange(0.02, 0.70, 0.01), 2)
costos = np.array([costo_total(y_tr, (p_oof >= u).astype(int)) / len(y_tr) for u in umbrales])
umbral_opt = float(umbrales[costos.argmin()])
print(f"Umbral óptimo (fuera de fold): {umbral_opt:.2f} · costo {costos.min():.3f} por pedido "
      f"· teórico C_FP/(C_FP+C_FN) = {C_FP / (C_FP + C_FN):.2f}")

fig3c, ax3c = plt.subplots(figsize=(7, 3.5))
ax3c.plot(umbrales, costos)
ax3c.axvline(umbral_opt, color="tab:red", linestyle="--", label=f"óptimo ({umbral_opt:.2f})")
ax3c.axhline(_costo_siempre, color="gray", linestyle=":", label="avisar siempre")
ax3c.set_xlabel("umbral de probabilidad para avisar"); ax3c.set_ylabel("costo por pedido (fuera de fold)")
ax3c.set_title("Con un falso negativo 10 veces más caro, conviene avisar desde p ≈ 0,08")
ax3c.legend()

modelo_final.fit(X_tr, y_tr)
resultado_test = evaluar_test(modelo_final, umbral_opt)          # UNA sola vez
# Misma pasada por el test, solo para reportar la comparación (no se elige nada con esto):
p_te = modelo_final.predict_proba(X_te)[:, 1]
comparacion = pd.Series({
    "nunca avisar": costo_total(y_te, np.zeros(len(y_te), dtype=int)) / len(y_te),
    "avisar siempre": costo_total(y_te, np.ones(len(y_te), dtype=int)) / len(y_te),
    f"{_elegido}, umbral 0,5": costo_total(y_te, (p_te >= 0.5).astype(int)) / len(y_te),
    f"{_elegido}, umbral {umbral_opt:.2f}".replace(".", ","): resultado_test["costo_por_pedido"],
}, name="costo por pedido (test)")
display(pd.Series(resultado_test).round(3))
display(comparacion.round(3))
verificar_ej3_extra()
```

El umbral que minimiza el costo fuera de fold es **0,08** (cerca del teórico 0,09), con **0,601** por pedido. En el **test** (1.200 pedidos, usado una vez): sensibilidad **0,825**, precisión **0,199** y costo **0,579** por pedido. Con el umbral 0,5 la misma logística casi no avisa (sensibilidad 0,058, costo 1,083, casi igual a "nunca avisar", 1,142). Con el umbral por costo baja a 0,579: **−49 %** frente a "nunca avisar" y **−35 %** frente a "avisar siempre" (0,886). Avisa a ~4 clientes por cada retraso real, y eso conviene porque cada mensaje de más cuesta 1 y cada retraso no avisado cuesta 10. PR-AUC en el test: 0,281; ROC-AUC: 0,766.

El umbral se eligió con predicciones **fuera de fold** del entrenamiento, no con el test: si se eligiera mirando el test, el 0,579 dejaría de ser una estimación honesta.

</details>

---
## Ejercicio 4 — Caza de la filtración (≈30 min)

Un compañero propone una variable nueva: la **tasa de retraso de cada comercio** (`tasa_comercio`), "porque hay comercios que siempre se atrasan". La agregó y evaluó un bosque con validación cruzada. El resultado salta de ≈ 0,74 a más de 0,85 de ROC-AUC.

Ver `lab11-guia-evaluacion.md`, §7.

### 4.1 Primero, el error
La evaluación del compañero, tal cual.

In [ ]:
com_tr = df.loc[i_tr, "id_comercio"].values      # comercio de cada pedido del entrenamiento
y_arr = y_tr.values


def cv_filtrado(modelo):
    """Agrega la tasa de retraso por comercio como variable y evalúa con validación cruzada."""
    tasa = pd.Series(y_arr).groupby(com_tr).transform("mean").values
    Xf = X_tr.copy(); Xf["tasa_comercio"] = tasa
    return cross_validate(modelo, Xf, y_tr, cv=CV, scoring="roc_auc", n_jobs=-1)["test_score"]


rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=1, random_state=SEED, n_jobs=-1)
auc_f = cv_filtrado(rf)
print(f"ROC-AUC con tasa_comercio: {auc_f.mean():.3f} ± {auc_f.std():.3f}")

*¿Se ve bien? ¿Qué tendría que pasar para que sea creíble?*

### 4.2 ✏️ TU TURNO

1. `paso_que_filtra` = la letra del paso que deja entrar información de la validación:
   - a) la partición estratificada de los folds;
   - b) la tasa de retraso por comercio calculada con todo el entrenamiento, incluida la fila que se evalúa;
   - c) el número de árboles del bosque.
2. Escriban `cv_honesto(modelo)`: un bucle sobre `CV.split(X_tr, y_tr)` en el que, **dentro de cada fold**, la tasa por comercio se calcula **solo con las filas de entrenamiento del fold**; con esa tasa se codifican entrenamiento y validación, y los comercios que no aparezcan en el entrenamiento del fold se rellenan con la tasa global del fold. Devuelve un arreglo con los 5 ROC-AUC.
3. `auc_filtrado = cv_filtrado(rf)`, `auc_honesto = cv_honesto(rf)` y `auc_sin_variable` = el mismo `rf` en `CV` **sin** la variable (`cross_validate(..., scoring="roc_auc")["test_score"]`).
4. En la celda de texto de abajo: una frase de por qué bajó el número, y si la variable vale la pena.
5. `conclusion_ej4` = `"a"`, `"b"` o `"c"`:
   - a) La filtración inflaba el resultado: ajustada solo con el entrenamiento, la variable no aporta y conviene dejarla fuera.
   - b) `tasa_comercio` es la variable más útil: sube el ROC-AUC de 0,74 a 0,86 y hay que incluirla.
   - c) El número cae porque el bosque con `min_samples_leaf=1` sobreajusta; con más árboles el resultado filtrado sería creíble.

Pistas: `pd.Series(y_arr[a]).groupby(com_tr[a]).mean()` da la tasa por comercio con las filas `a` · `pd.Series(com_tr[b]).map(tasas).fillna(global_)` · `roc_auc_score(y_tr.iloc[b], modelo.predict_proba(Xb)[:, 1])`.

In [ ]:
# ✏️ TU TURNO — completen (descomenten y reemplacen los ...)
# paso_que_filtra = ...
#
# def cv_honesto(modelo):
#     aucs = []
#     for a, b in CV.split(X_tr, y_tr):
#         tasas = ...                  # tasa por comercio, SOLO con las filas a
#         global_ = ...                # tasa global, SOLO con las filas a
#         Xa, Xb = X_tr.iloc[a].copy(), X_tr.iloc[b].copy()
#         Xa["tasa_comercio"] = ...
#         Xb["tasa_comercio"] = ...    # comercios que no están en a → global_
#         modelo.fit(Xa, y_tr.iloc[a])
#         aucs.append(...)
#     return np.array(aucs)
#
# auc_filtrado = cv_filtrado(rf)
# auc_honesto = cv_honesto(rf)
# auc_sin_variable = ...
# conclusion_ej4 = ...

✏️ **Respuesta (4):** ¿por qué bajó el número? ¿Vale la pena la variable? *(escriban aquí)*

### ✅ Verificación automática
Debe imprimir 6 ✓.

In [ ]:
def verificar_ej4():
    checks = [
        ("paso_que_filtra es el correcto",
         lambda: paso_que_filtra == "b"),
        ("auc_filtrado ≈ 0,855 (±0,03)",
         lambda: abs(auc_filtrado.mean() - 0.855) <= 0.03),
        ("auc_honesto ≈ 0,689 (±0,04)",
         lambda: abs(auc_honesto.mean() - 0.689) <= 0.04),
        ("la versión honesta queda al menos 0,08 bajo la filtrada",
         lambda: auc_filtrado.mean() - auc_honesto.mean() >= 0.08),
        ("auc_sin_variable ≈ 0,737 (±0,03)",
         lambda: abs(auc_sin_variable.mean() - 0.737) <= 0.03),
        ("conclusion_ej4 es la correcta",
         lambda: conclusion_ej4 == "a"),
    ]
    return ejecutar_checks(checks)

verificar_ej4()

<details>
<summary><b>Solución propuesta (Ejercicio 4)</b> — clic para desplegar</summary>

```python
paso_que_filtra = "b"


def cv_honesto(modelo):
    """BIEN: dentro de cada fold, la tasa se aprende solo con el entrenamiento del fold."""
    aucs = []
    for a, b in CV.split(X_tr, y_tr):
        tasas = pd.Series(y_arr[a]).groupby(com_tr[a]).mean()
        global_ = y_arr[a].mean()
        Xa, Xb = X_tr.iloc[a].copy(), X_tr.iloc[b].copy()
        Xa["tasa_comercio"] = pd.Series(com_tr[a]).map(tasas).values
        Xb["tasa_comercio"] = pd.Series(com_tr[b]).map(tasas).fillna(global_).values
        modelo.fit(Xa, y_tr.iloc[a])
        aucs.append(roc_auc_score(y_tr.iloc[b], modelo.predict_proba(Xb)[:, 1]))
    return np.array(aucs)


auc_filtrado = cv_filtrado(rf)
auc_honesto = cv_honesto(rf)
auc_sin_variable = cross_validate(rf, X_tr, y_tr, cv=CV, scoring="roc_auc", n_jobs=-1)["test_score"]
for nombre, a in [("filtrada", auc_filtrado), ("honesta", auc_honesto), ("sin la variable", auc_sin_variable)]:
    print(f"{nombre:>16}: {a.mean():.3f} ± {a.std():.3f}")
conclusion_ej4 = "a"
verificar_ej4()
```

**Dónde filtra:** la tasa de cada comercio se calculó con **todo** el entrenamiento, **antes** de la validación cruzada. Cuando un pedido cae en el fold de validación, su propia etiqueta (`retraso`) ya está dentro de la tasa de su comercio. Con ≈ 9,6 pedidos de entrenamiento por comercio, una sola etiqueta mueve la tasa ≈ 10 puntos: el modelo "ve" la respuesta a través de la variable. El bosque con hojas de 1 pedido la aprovecha al máximo.

**Corregido:** filtrada **0,855 ± 0,021** → honesta **0,689 ± 0,021**. El número bajó 0,17 y **es el real**. Además, la versión honesta queda **por debajo** de no usar la variable (**0,737**): con tan pocos pedidos por comercio, la tasa aprendida en el fold es casi puro ruido y confunde al modelo. Conclusión: la variable no aporta y conviene dejarla fuera (o, si se quisiera usar, suavizarla y validarla con `GroupKFold` por comercio; guía §7).

Regla general: **todo paso que aprende del dato** (imputar, escalar, seleccionar variables, codificar por la tasa del objetivo) va **dentro** de la validación: dividir → ajustar con el entrenamiento → aplicar a la validación. La Clase 12 lo automatiza con `Pipeline`.

</details>

In [ ]:
# (Celda del profesor / autocomprobación del notebook)
def _cv_honesto(modelo):
    aucs = []
    for a, b in CV.split(X_tr, y_tr):
        tasas = pd.Series(y_arr[a]).groupby(com_tr[a]).mean()
        Xa, Xb = X_tr.iloc[a].copy(), X_tr.iloc[b].copy()
        Xa["tasa_comercio"] = pd.Series(com_tr[a]).map(tasas).values
        Xb["tasa_comercio"] = pd.Series(com_tr[b]).map(tasas).fillna(y_arr[a].mean()).values
        modelo.fit(Xa, y_tr.iloc[a])
        aucs.append(roc_auc_score(y_tr.iloc[b], modelo.predict_proba(Xb)[:, 1]))
    return np.array(aucs)


_auc_filtrado = auc_f
_auc_honesto = _cv_honesto(clone(rf))
assert _auc_filtrado.mean() - _auc_honesto.mean() >= 0.08, "La solución de referencia del Ej. 4 no pasa la verificación"
print(f"filtrada {_auc_filtrado.mean():.3f} → honesta {_auc_honesto.mean():.3f}")

---
## 6 · Ahora con SU proyecto — `04-modelado.ipynb` (11:35–12:05)

Lo que sigue no es un ejercicio: es la plantilla de la **sección 6 del informe** (Informes 3 y 4, criterios II.1 a II.6 de la pauta). Creen `notebooks/04-modelado.ipynb` en su repositorio y copien las dos celdas de abajo. Lo que entra al modelo es lo que marcaron como `entra` en la **ficha de candidatas** (sección 5.3 del informe, Lab 09).

Mientras trabajan, pasa la **ronda de consulta**. Recorran la tabla de auditoría punto por punto: `cumple` (sí / no / parcial) y `que_cambio_hago`. Lo que no alcancen a corregir hoy, se declara como **limitación** en el informe.

In [ ]:
# ==========================================================
# 04-modelado.ipynb (Clase 11) — Sección 6 del informe — completar con SU proyecto
# ==========================================================
# 0. Variables que ENTRAN: leer data/processed/ficha_candidatas.csv (sección 5.3 del informe).
#    Si todavía no existe la ficha, se escribe primero (Lab 09, sección 9.4): lo que entra = lo que se modela.
#    Ninguna variable con fuga marcada como `entra`.
#
# 6.1 Diseño de la evaluación                                         (II.1 · 20 pts)
#    - tipo de problema (clasificación / regresión)
#    - UNA FRASE: qué error es más caro, y por qué  ->  de ahí sale la métrica principal (+ secundarias)
#    - baseline trivial (DummyClassifier / DummyRegressor) CON su valor, y la política trivial más exigente
#    - partición que respeta la estructura del dato:
#         mismo individuo/unidad repetida -> GroupKFold · tiempo -> TimeSeriesSplit · clase rara -> StratifiedKFold
#    - TEST separado AL INICIO (train_test_split con semilla) y usado UNA sola vez, al final
#
# 6.2 Modelos                                                          (II.2 · 20 pts)
#    - >= 2 modelos de FAMILIAS distintas (lineal, árbol, ensamble, vecinos...), justificados para ESTE problema
#    - sobre las variables que entran en la ficha; toda diferencia con la ficha se explica
#    - todo paso que aprende de los datos (imputar, escalar, seleccionar, codificar) se ajusta SOLO con el
#      entrenamiento de cada partición  (hoy: a mano o con make_pipeline como en el lab; Clase 12: Pipeline)
#
# 6.3 Validación cruzada e hiperparámetros                             (II.3 · 25 pts)
#    - MISMAS particiones para todos los modelos; tabla modelo · media ± desviación
#    - modelo principal: >= 1 hiperparámetro de complejidad con curva de validación o búsqueda
#      (declarar: espacio, n.º de combinaciones, métrica, semilla; revisar bordes; búsqueda dentro del entrenamiento)
#
# 6.4 Comparación, elección y limitaciones                             (II.4 · 15 pts)
#    - tabla: modelo · métrica · media ± desv. · vs. baseline · hiperparámetros · veredicto
#    - empate práctico (diferencia < desviación entre folds) -> gana el más simple
#    - modelo elegido en UNA frase; resultado en el TEST (una vez), en unidades reales
#    - limitaciones declaradas
#
# Reproducibilidad (II.5 · 10 pts): semilla fijada, corre desde clon vacío, cada cifra cita su celda.

In [ ]:
auditoria = pd.DataFrame({
    "punto": [
        "La métrica está justificada por el costo del error (una frase)",
        "Hay baseline trivial, con su valor",
        "Mismas particiones para todos los modelos",
        "Ningún paso aprende con datos de validación o de prueba",
        "Los hiperparámetros tienen espacio declarado, n.º de combinaciones, métrica y semilla",
        "El test se usó una sola vez, con el modelo ya elegido",
        "El modelo elegido tiene una frase de justificación",
    ],
    "criterio_pauta": ["II.1", "II.1", "II.3", "II.2", "II.3", "II.1 / II.4", "II.4"],
    "cumple": ["", "", "", "", "", "", ""],          # completar: sí / no / parcial
    "que_cambio_hago": ["", "", "", "", "", "", ""],  # completar
})
auditoria

Dos recordatorios para la ronda de consulta:

- La **trazabilidad** se revisa por muestreo: se eligen **6 cifras de la sección 6** (1 del baseline, 2 de la validación cruzada, 2 de hiperparámetros y 1 del test) y cada una tiene que poder rastrearse hasta la celda que la produce. Una **cifra central sin celda** gatilla el gate.
- El repositorio debe correr los **cuatro notebooks en orden desde un clon vacío** (`git clone` → `pip install -r requirements.txt` → Restart & Run All de cada uno).

Ejemplo de una fila de la tabla de comparación bien escrita (con las cifras del lab):

| modelo | métrica (por qué) | media ± desv. (CV, 5 folds) | vs. baseline | hiperparámetros | veredicto |
|---|---|---|---|---|---|
| Regresión logística | PR-AUC (el FN cuesta 10 veces el FP) | 0,352 ± 0,027 | +0,238 sobre 0,114 | escalado + `max_iter=1000` (sin búsqueda) | **elegido** (empate con el bosque, 0,337 ± 0,025; es el más simple) |

---
### Checklist de salida de la Clase 11

- [ ] Ejercicios 1, 2 y 3 pasan su verificación automática (el 4 y el extra 3.3 pueden quedar para la casa)
- [ ] `notebooks/04-modelado.ipynb` con la estructura 6.1–6.4 (o al menos 6.1 y 6.2 hoy)
- [ ] Tabla de auditoría completa: cada punto con `cumple` y `qué cambio hago`
- [ ] Lo que no se alcanza a corregir hoy, anotado como limitación para el informe
- [ ] Informes 3 y 4 (documento único): **domingo 8-nov, 23:59, AULA-USM** (−10 por día de atraso, máx. 3 días, aplicado a las dos notas; rondas de consulta también en las Clases 12 y 13)
- [ ] Para la Clase 12 (29-oct): `04-modelado.ipynb` en un estado que corra de principio a fin